# MTP speculative decoding in Rapid-LLM

**Presentation outline · 10 content slides · 12 minutes + 3 minutes for questions**

**Research question.** When does verifying several proposed tokens in one backbone pass reduce decode latency, and how much extra GPU memory does it require?

| Section | Slides | Time |
|---|---:|---:|
| Concept | 1–3 | 3:00 |
| Rapid-LLM changes and assumptions | 4–7 | 4:30 |
| Results and comparisons | 8–10 | 4:30 |

Use the **On slide** bullets for slide copy. Use the **Speaker notes** to rehearse. Slides 8 and 9 use completed simulator runs.

## Slide 1 · Why decode one token at a time? · 1:00

**On slide**

- Ordinary autoregressive decode advances one token per target-model pass.
- The next token normally depends on the token just generated.
- Question for this project. Can one expensive backbone pass advance several output tokens?

**Visual.** Draw three sequential target calls with one token emerging from each.

**Speaker notes.** Spend one sentence distinguishing prompt prefill from decode. The results focus on decode latency per emitted token. Introduce the memory tradeoff before discussing why speculation can help.

## Slide 2 · Draft sequentially, verify together · 1:00

**On slide**

- $D=3$ MTP modules propose three tokens in sequence.
- Each guess uses a $2H \rightarrow H$ projection, a transformer layer, and a shared output head.
- The backbone scores $D+1=4$ query positions together. Keep the consecutive accepted guesses and emit one additional token.

**Visual.** Show three small draft blocks feeding one wide verification block. Mark the case where guess 1 passes and guess 2 fails, yielding two output tokens that round.

**Speaker notes.** The proposed token IDs are available before verification, so the backbone can score their positions in one causal forward pass. If all three guesses pass, the fourth output token comes from the verifier's next-token distribution.

## Slide 3 · Acceptance determines how far a round advances · 1:00

**On slide**

- Use conditional acceptance probabilities `[0.9, 0.8, 0.7]`.
- Expected tokens per round $=1+p_1+p_1p_2+p_1p_2p_3=3.124$.
- The round always advances by at least one token.

**Visual.** Use four bars labeled “at least 1”, “at least 2”, “at least 3”, and “4”, with heights 1.000, 0.900, 0.720, and 0.504.

**Speaker notes.** To reach the third guess you must pass the first two, so its unconditional contribution is $0.9\times0.8\times0.7$. The simulator consumes acceptance probabilities supplied in the YAML.

## Slide 4 · How MTP enters Rapid-LLM · 1:00

**On slide**

- YAML supplies guess count, module count, layers per module, sharing, and acceptance probabilities.
- `config.py` validates the settings. `simulate_inference_graph.py` drives the existing decode-context sampling and integration.
- Positive guess counts use the MTP timing path. Zero guesses use ordinary decode.

**Visual.** Draw the pipeline from YAML to sampled contexts to latency and peak-memory reports.

**Speaker notes.** Explain the modeling scope in one sentence. The simulator prices operations from shapes and hardware parameters. It does not train MTP weights or generate token IDs.

## Slide 5 · Price the draft and verifier separately · 1:30

**On slide**

- **Draft.** Sum the costs of the sequential projections, single-position transformer blocks, and output-head calls.
- **Verify.** Reuse the backbone decode graph once with $D+1$ queries. Price the attention block against the longest context in the round.
- **Amortize.** Divide draft plus verifier time by expected tokens per round, then integrate the per-token estimate across sampled contexts.

**Visual.** At the first sampled context, show a stacked bar with **2.98 ms draft + 12.33 ms verify = 15.31 ms per round**, then divide by 3.124 to get about **4.90 ms per output token**.

**Speaker notes.** Point to `inference_timing.py` and `llm_util.py`. The main GEMMs have more query rows during verification, so the model prices them together. The verifier cost is neither assumed to be free nor computed as four separate one-token passes.

## Slide 6 · Count what lives on the GPU · 1:00

**On slide**

- Charge each MTP module for its projection weights, transformer weights, and KV cache.
- Reuse the backbone embedding and output head with the supplied sharing flag.
- Include in-flight verification positions when estimating decode memory.

**Visual.** Draw the backbone allocation and the added MTP projection, layer-weight, and cache blocks. Show the shared output head once.

**Speaker notes.** Point to `memory_estimation.py`. Shared weights still incur compute when the head is called. The memory model reserves a full-context draft cache. Draft workspace and cache initialization time are outside the modeled scope.

## Slide 7 · Establish the baseline and scope · 1:00

**On slide**

- Dense MiMo-7B-shaped backbone, analytical A100 80 GB, batch size 1.
- Same 1,024-token prompt and 1,024-token generation for baseline and MTP runs.
- Explicit $D=0$ and an omitted MTP block produced byte-identical timing and memory reports.

**Visual.** Show the matching baseline and explicit-zero configurations side by side, highlighting the one verification position and one output token per step.

**Speaker notes.** Mention that normalization by expected round progress is a steady-state approximation near the end of generation. The figures are simulation estimates for this workload and hardware configuration.

## Slide 8 · The default configuration reduces decode latency · 1:30

| Configuration | Mean decode time per output token | Peak GPU memory |
|---|---:|---:|
| Ordinary decode | 12.33 ms | 14.63 GiB |
| Three MTP guesses | 4.92 ms | 15.82 GiB |

**On slide.** Call out **2.51× modeled decode speedup** and **+1.19 GiB peak memory**. Show separate charts for latency and memory because the units differ.

**Speaker notes.** Tie the result to slide 5. Each round costs more than one ordinary decode step, but advances 3.124 tokens in expectation. Quote decode-only speedup and label the memory result as overall peak. These are completed simulator runs.

**Evidence.** [MTP timing report](results/mtp/LLM_inference_results.txt) · [Baseline timing report](results/baseline/LLM_inference_results.txt) · [Baseline memory report](results/baseline/memory-summary/memory_capacity_comparison.txt)

## Slide 9 · Test how sensitive the speedup is · 2:00

**Completed ablations.** The runs use the same hardware, batch, and token lengths as slide 8. One and two guesses give 7.02 and 5.48 ms/token. At three guesses, low acceptance gives 13.82 ms/token and ideal acceptance gives 3.84 ms/token.

| Sweep | Settings | Readout |
|---|---|---|
| Guess count | $D=0,1,2,3$, using the first $D$ probabilities from `[0.9, 0.8, 0.7]` | Mean decode latency and peak memory versus $D$ |
| Acceptance | Keep $D=3$, compare low `[0.1, 0.1, 0.1]`, default `[0.9, 0.8, 0.7]`, and ideal `[1, 1, 1]` | Latency relative to baseline and expected tokens per round |

**Visual.** Draw latency versus guess count with a horizontal ordinary-decode line. Use a second panel for peak memory versus guess count. Present acceptance as a small inset or table.

**Speaker notes.** At fixed $D$, the model prices the same draft and verification work across acceptance settings. Acceptance changes how much progress the round buys. Ask whether the added guesses pay for themselves under the tested probabilities. Point out that the low-acceptance run is slower than the baseline.

## Slide 10 · Explain the tradeoff and close · 1:00

**On slide**

- Report the speed and memory result for the default workload.
- The break-even condition is
  $$
  \mathbb{E}[\text{output tokens per round}]
  >
  \frac{T_{\text{draft}}+T_{\text{verify}}}{T_{\text{normal decode step}}}.
  $$
- Compare the low and default acceptance cases against break-even.

**Speaker notes.** Explain that the verifier cost and draft overhead set how much progress is needed to break even. Restate that the project covers analytical single-GPU timing and memory. Leave three minutes for questions.

## Q&A notes and presentation checklist

These notes sit outside the 10-slide deck.

- **Why four positions for three guesses?** The verifier scores the three candidates and can supply one additional token when they all pass.
- **Why does the head cost time if it is shared?** Sharing reuses parameter memory. Every invocation still performs the vocabulary projection.
- **Is 2.51× a measured GPU speedup?** It is estimated from the Rapid-LLM analytical model for the supplied single-A100, batch-1 workload.
- **Why reserve the full draft KV cache?** The current memory estimate assumes the MTP layers retain context-sized caches. It also makes room for in-flight verification positions.
- **Before presenting.** Review the saved slide 9 reports, practice the 12-minute talk, and keep the slide count at 10. The course screenshots list a November 30, 11:59 p.m. submission deadline and a December 3 presentation slot.

[Rapid-LLM project README](../README.md) · [MTP configuration](../source/configs/model-config/MiMo-7B_mtp.yaml)